<a href="https://colab.research.google.com/github/Hashim123132/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashim123132/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order**, each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the `building-baselines` skill + `flyrank/flyrank-data`.

Lane 1 (Ranking Signal Analysis) on the starter slice (30,000 rows, one row per content item). Two signals are checked first, then ONE rule is encoded, ranked, and reviewed. No warehouse needed.

In [ ]:
import pandas as pd, numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")
df["down"] = (df.trend_direction == "down").astype(int)   # eval only, never a rule input
print("rows:", len(df), "| clients:", df.client_id.nunique())

rows: 30000 | clients: 32


## 1. My rule and its reason codes

### Signal A — Staleness → decline

I tested whether older content is more likely to show decline, using `content_age_days` and a minimum evidence threshold of `impressions_90d >= 500`.

| Age band |     n | Down rate |
| -------- | ----: | --------: |
| 0–90d    |   221 |     0.692 |
| 90–180d  | 6,613 |     0.677 |
| 180–365d | 6,009 |     0.600 |
| 365+     | 3,883 |     0.443 |

**Verdict: OPPOSITE.**

In this slice, younger pages had higher observed decline rates than older pages. Therefore, I do not use staleness as a positive signal in my rule.

### Signal B — CTR relative to position

I tested whether CTR varies systematically with `position_tier`, using the same `impressions_90d >= 500` evidence floor.

The observed median CTR was:

`page_1 = 0.24`, `striking = 0.17`, `page_3_5 = 0.09`, `deep = 0.00`.

**Verdict: CONFIRMED.**

CTR is higher for pages in stronger ranking tiers. This supports looking for pages whose CTR is below the typical CTR for their ranking tier.

### Rule in plain words

I will prioritize pages that have enough search visibility, rank in a relatively visible position, and have a CTR below the typical CTR for their position tier.

### Reason code

`decline_risk_visible_page`

### Action label

`review`


In [ ]:
# Signal A: staleness vs down rate (bucket table with n), volume floor applied
vA = df[df.impressions_90d >= 500].copy()
vA["age_band"] = pd.cut(vA.content_age_days, bins=[0, 90, 180, 365, 1000], right=True)
tA = vA.groupby("age_band", observed=True).agg(n=("content_age_days", "size"),
                                              down_rate=("down", "mean")).round(3)
print("Signal A: down rate by content_age_days (impressions >= 500)")
print(tA)

Signal A: down rate by content_age_days (impressions >= 500)
                n  down_rate
age_band                    
(0, 90]       221      0.692
(90, 180]    6613      0.677
(180, 365]   6009      0.600
(365, 1000]  3883      0.443


In [ ]:
# Signal B: median CTR by position tier (bucket table with n), volume floor + real position
vB = df[(df.impressions_90d >= 500) & (df.avg_position > 0)].copy()
tB = vB.groupby("position_tier").agg(n=("ctr", "size"), median_ctr=("ctr", "median")).round(3)
print("Signal B: median CTR by position_tier (impressions >= 500, avg_position > 0)")
print(tB)

Signal B: median CTR by position_tier (impressions >= 500, avg_position > 0)
                  n  median_ctr
position_tier                  
deep            389        0.00
page_1         7064        0.24
page_3_5       4330        0.09
striking       4485        0.17
top_3           458        0.20


## 2. Build the ranked queue (writes the CSV)

**Score** = `(avg_position <= 20)` × `(tier median CTR - page CTR)` clipped at 0 × `log1p(impressions_90d)`.

Row gates: `impressions_90d >= 500` and `avg_position > 0` (position 0 = no data, not rank zero).
Inputs are all observed signals, knowable before any decision: impressions, position, CTR. No `trend_*`, no label, no future window.

The ranked queue is written to `work/outputs/baseline_action_score.csv` (regenerated every run; stays out of git by design).

Honest check next to the score: precision@K on the top of the queue vs the base rate.

In [ ]:
import os

v = df[(df.impressions_90d >= 500) & (df.avg_position > 0)].copy()
tier_ctr = v.groupby("position_tier").ctr.transform("median")
v["gap"] = (tier_ctr - v.ctr).clip(lower=0)
v["score"] = (v.avg_position <= 20).astype(int) * v.gap * np.log1p(v.impressions_90d)

v["reason_code"] = "decline_risk_visible_page"
v["action"] = "review"

queue = v.sort_values("score", ascending=False)[
    ["content_id", "client_id", "position_tier", "avg_position", "ctr",
     "impressions_90d", "clicks_90d", "gap", "score", "reason_code", "action"]
].copy()

os.makedirs("work/outputs", exist_ok=True)
out_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(out_path, index=False)
print("wrote", out_path, "| rows:", len(queue), "| flagged (score>0):", int((queue.score > 0).sum()))

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

labels = v.loc[queue.index, "down"].to_numpy()
print("base rate (random pick):", round(labels.mean(), 3))
print("precision@10:", round(precision_at_k(queue.score, labels, 10), 3))
print("precision@50:", round(precision_at_k(queue.score, labels, 50), 3))

wrote work/outputs/baseline_action_score.csv | rows: 16726 | flagged (score>0): 5885
base rate (random pick): 0.596
precision@10: 0.7
precision@50: 0.68


## 3. Top-20 review

For each of the top 20: the action, why it is there, and what would make it wrong.

| # | content_id | action | why it is there | what would make it wrong |
|---|---|---|---|---|
| 1 | content_c8e9d6ab9013 | review | page_1 (pos 9.7), 208k impressions, CTR 0.00, no clicks in window | title/meta may be fine and the SERP itself is broken; zero clicks with 208k impressions can be a tracking or intent mismatch, not decay |
| 2 | content_453722754fea | review | page_1 (pos 7.6), 140k impressions, CTR 0.01, 16 clicks | page is only 97 days old; the gap may be a real intent mismatch rather than decay risk |
| 3 | content_39881853ef0c | review | page_1 (pos 7.2), 112k impressions, CTR 0.01 | same intent-mismatch alternative; two top picks on one client suggest client-level SERP pattern, not per-page decay |
| 4 | content_c84a0ab98e90 | review | page_1 (pos 7.8), 223k impressions, CTR 0.03, 70 clicks | huge demand, low CTR, but NOT currently down; refresh could hurt a page that is holding |
| 5 | content_0919dd345d80 | review | page_1 (pos 7.0), 119k impressions, CTR 0.02 | pos 7.0 with CTR 0.02 sits far under tier median; could be a title/snippet problem or a query that never meant to convert |
| 6 | content_d274ac4158ef | review | page_1 (pos 6.8), 65k impressions, CTR 0.01, 6 clicks | not currently down; low clicks could be seasonality in the query rather than a fixable gap |
| 7 | content_e5f459e737b7 | review | page_1 (pos 5.9), 56k impressions, CTR 0.01, transactional intent | transactional page with near-zero CTR at pos 5.9: if the SERP shows competitors with better offers, no title fix helps |
| 8 | content_c1fe78bc4e37 | review | page_1 (pos 7.5), 134k impressions, CTR 0.03, commercial intent | commercial gap at pos 7.5; wrong if the page already targets the strongest sub-intent and the gap is price/offer, not text |
| 9 | content_339b357d04c7 | review | page_1 (pos 3.7), 47k impressions, CTR 0.01 | pos 3.7 is the strongest rank in the top 10; a near-zero CTR there is more likely an intent mismatch than decay, and n is small |
| 10 | content_65114d89496d | review | page_1 (pos 6.5), 73k impressions, CTR 0.02, 482 days old | older transactional page; wrong if impressions are inflated by a broad query the page never promised to answer |
| 11 | content_b115f7c74779 | review | page_1 (pos 8.0), 123k impressions, CTR 0.03, 37 clicks, transactional, 313 days old | not down; a transactional gap at pos 8.0 is often offer/price competition, so no text fix helps |
| 12 | content_65d9331f55fb | review | page_1 (pos 7.8), 66k impressions, CTR 0.02, 13 clicks | down, but the CTR gap is modest; wrong if the drop is a seasonal dip in the query, not the page |
| 13 | content_ca17a024f90c | review | page_1 (pos 9.1), 39k impressions, CTR 0.01, 4 clicks | pos 9.1 sits at the page-one edge; volume is small enough that the gap may be noise |
| 14 | content_63f88d16fdb8 | review | page_1 (pos 6.4), 99k impressions, CTR 0.03, 31 clicks, 419 days old | older page, down; wrong if the demand itself shrank, which editing cannot fix |
| 15 | content_f986bd514b6e | review | page_1 (pos 6.6), 22k impressions, CTR 0.00, 1 click, no intent recorded | no main_intent on record; a zero CTR with 1 click may be tracking or a low-relevance query, not decay |
| 16 | content_36ff89c8214e | review | page_1 (pos 7.3), 295k impressions, CTR 0.05, 154 clicks | huge demand, CTR 0.05 only moderately under tier, NOT down; the gap may be normal for the niche |
| 17 | content_d0cc5baa4995 | review | page_1 (pos 6.6), 84k impressions, CTR 0.03, 28 clicks, transactional | down; wrong if a competitor listing change took the clicks rather than anything on this page |
| 18 | content_f6ae0f36d70d | review | page_1 (pos 9.0), 45k impressions, CTR 0.02, 9 clicks, 95 days old | young page at pos 9.0; the gap may be normal ramp-up, not decay |
| 19 | content_825a9788af8d | review | page_1 (pos 5.6), 17k impressions, CTR 0.00, 0 clicks, 421 days old | pos 5.6 with zero clicks; wrong if impressions come from a broad head query the page only partially answers |
| 20 | content_8ba781dafa55 | review | page_1 (pos 9.0), 16k impressions, CTR 0.00, 0 clicks | smallest volume in the top 20; the gap is likely noise or a mismatch, the least confident pick |


In [ ]:
# Show the top-20 the review above refers to, so the numbers are visible in the notebook
top20 = queue.head(20)
print(top20[["content_id", "position_tier", "avg_position", "ctr", "impressions_90d", "score"]].to_string(index=False))


          content_id position_tier  avg_position  ctr  impressions_90d    score
content_c8e9d6ab9013        page_1           9.7 0.00           208678 2.939653
content_453722754fea        page_1           7.6 0.01           140079 2.725493
content_39881853ef0c        page_1           7.2 0.01           112434 2.674930
content_c84a0ab98e90        page_1           7.8 0.03           223271 2.586391
content_0919dd345d80        page_1           7.0 0.02           119217 2.571516
content_d274ac4158ef        page_1           6.8 0.01            65138 2.549384
content_e5f459e737b7        page_1           5.9 0.01            56363 2.516105
content_c1fe78bc4e37        page_1           7.5 0.03           134055 2.479263
content_339b357d04c7        page_1           3.7 0.01            46879 2.473730
content_65114d89496d        page_1           6.5 0.02            72631 2.462495
content_b115f7c74779        page_1           8.0 0.03           123469 2.461988
content_65d9331f55fb        page_1      

## 4. Weak picks + leakage check

**Weak picks:** rows 4, 6, 9 of the top 10 (content_c84a0ab98e90, content_d274ac4158ef, content_339b357d04c7) are flagged by the rule but are **not currently down**. The rule scores a CTR gap, and a gap can be an intent mismatch, a competitor's SERP change, or seasonality rather than decay. That is the rule's main false-positive mode: it ranks "under-capturing clicks" pages, and some of those are simply harder queries.

**Leakage check:** the rule's inputs are `avg_position`, `ctr`, `impressions_90d`, and the position tier median, all observed in the trailing 90-day window, knowable before any review decision. `down` (`trend_direction`) is used for evaluation only, never as an input. No product flags, no future-window columns. The check below asserts it.

In [ ]:
# Leakage check: rule inputs must not include label sources, product flags, or future windows
rule_inputs = {"avg_position", "ctr", "impressions_90d"}
banned = {"trend_direction", "trend_pct", "is_declining_label"}
print("rule inputs:", sorted(rule_inputs))
print("no label-derived inputs:", rule_inputs.isdisjoint(banned))
print("no product flags shipped in data (health_score etc.):", not any(
    c in df.columns for c in ["health_score", "priority_score", "action_type"]))

rule inputs: ['avg_position', 'ctr', 'impressions_90d']
no label-derived inputs: True
no product flags shipped in data (health_score etc.): True


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled, markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit your repo URL on the card. Done.